# Phase 2 — registered forward shadow v2

This notebook never tunes a model or strategy. Use `prepare` once for a new 720-hour block. After Codex commits and registers the exact manifest, use `score` after refreshing notebooks 01–02. Labels are not consumed by scoring.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = 'codex/phase1-research-v2'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, REPO_DIR)
print('Repository branch:', REPO_BRANCH)
print('Repository commit:', repo_commit)
print('Restart the session after a changed checkout before trusting old imports.')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)

In [ ]:
from pathlib import Path

# First run: 'prepare'. Codex will inspect/register its exact manifest.
# Later collection runs: 'score'. Do not use score before registration exists.
ACTION = 'prepare'  # 'prepare' or 'score'
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot')
BLOCK_DIR = DRIVE_BASE / 'forward_shadow_v2' / 'current_prepared_block'
REGISTRATION = BLOCK_DIR / 'forward_shadow_registration.json'
LEDGER = BLOCK_DIR / 'forward_shadow_ledger.jsonl'
LABELED = DRIVE_BASE / 'data' / 'processed' / 'labeled_1h.parquet'
FEATURES = DRIVE_BASE / 'data' / 'processed' / 'features_1h.parquet'
OOF_TARGETS = DRIVE_BASE / 'research_inputs' / 'forward_shadow_v2' / 'oof_opportunity_targets.csv'
print('Action:', ACTION)
print('Block directory:', BLOCK_DIR)

In [ ]:
if ACTION == 'prepare':
    from yenibot.automation.phase2_forward_shadow_prepare import main as run_prepare
    if BLOCK_DIR.exists():
        raise FileExistsError(f'Prepared block already exists; do not overwrite it: {BLOCK_DIR}')
    run_prepare([
        '--labeled', str(LABELED),
        '--oof-targets', str(OOF_TARGETS),
        '--output-dir', str(BLOCK_DIR),
        '--config', f'{REPO_DIR}/config.yaml',
        '--spec', f'{REPO_DIR}/configs/forward_shadow_v2.json',
        '--repo-dir', REPO_DIR,
    ])
elif ACTION == 'score':
    from yenibot.automation.phase2_forward_shadow_score import main as run_score
    if not REGISTRATION.exists():
        raise FileNotFoundError('Codex must commit and register the exact manifest before scoring.')
    run_score([
        '--features', str(FEATURES),
        '--block-dir', str(BLOCK_DIR),
        '--registration', str(REGISTRATION),
        '--ledger', str(LEDGER),
        '--config', f'{REPO_DIR}/config.yaml',
    ])
else:
    raise ValueError("ACTION must be exactly 'prepare' or 'score'")